# sweep-v1 -- settings and one new rule on top of eat-rest-v1 (baseline untouched)

Mechanism study, not a scored experiment: nothing is written to `results/`. The preserved baseline
file is never edited; variants are built by passing keyword overrides to its policy class
(`external/candidates/sweep_config.py`), and the new rule lives in its own candidate folder
(`external/candidates/eat-rest-fedbirth`, which loads the baseline unmodified and subclasses it).

Why these knobs: a death diagnosis of v1 on validation seeds 2001/2005/2006/2010 showed (a) the population
is capped at ~6 young agents shrinking to a floor of 2, so the game ends within seconds once 3-4 agents have
a bad streak, (b) late-game newborns (75 energy, ~20 s of fuel) starve at age 13-27 while adults hold 200+
energy, (c) predators kill 30-77 agents per game throughout.

Progress lines appear as games finish: `name: mean s (n games, paired delta vs the first variant)`.
Noise warning: paired differences on 16 seeds have a standard error of ~60-80 s. Treat anything under
~+150 s as unproven and confirm on the second seed block before believing it.

**Cluster setup:** same as `test-eat-rest-v1.ipynb` (`.env` with `GITHUB_TOKEN=<token>`).
If `rl-v1` training is still running it is using all 40 CPUs; interrupt it first or these games crawl.

In [ ]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}

In [ ]:
import glob
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

# Must run from survival-simulator/ so `src`, `agents`, `training` import.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])

print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

In [ ]:
import json, subprocess, sys, time
from collections import defaultdict
import pandas as pd

def sweep(candidate, variants, seeds, out, workers=None, every=8):
    """Run external/candidates/sweep_config.py and print a running table as games finish.
    The first variant is the reference; 'delta' is the paired mean difference on seeds both have finished."""
    cmd = [sys.executable, "-u", "external/candidates/sweep_config.py", "--candidate", candidate,
           "--seeds", *seeds, "--out", out, *[x for v in variants for x in ("--variant", v)]]
    if workers: cmd += ["--workers", str(workers)]
    print("running:", " ".join(cmd)); t0 = time.perf_counter()
    names = [v.split(":")[0] for v in variants]; done = defaultdict(dict); n = 0
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if not line.startswith("{"):
            if "pkg_resources" not in line and "pygame" not in line: print(line, end="")
            continue
        row = json.loads(line); done[row["variant"]][row["seed"]] = row["extinction_time"]; n += 1
        if n % every == 0:
            ref = done[names[0]]
            parts = []
            for name in names:
                t = done[name]
                if not t: continue
                both = [t[s] - ref[s] for s in t if s in ref]
                parts.append(f"{name}: {sum(t.values())/len(t):.0f} s (n={len(t)}" + (f", delta {sum(both)/len(both):+.0f}" if both and name != names[0] else "") + ")")
            print(f"[{(time.perf_counter()-t0)/60:4.1f} min, {n} games]  " + " | ".join(parts))
    proc.wait()
    if proc.returncode: raise RuntimeError(f"sweep failed with exit code {proc.returncode}")
    return pd.read_csv(out)

def table(df):
    wide = df.pivot(index="seed", columns="variant", values="extinction_time")
    ref = wide.columns[0] if "base" not in wide.columns else "base"
    summary = pd.DataFrame({"mean": wide.mean(), "median": wide.median(), "min": wide.min(), "max": wide.max(),
                            "delta_vs_" + ref: wide.sub(wide[ref], axis=0).mean(),
                            "se_of_delta": wide.sub(wide[ref], axis=0).sem(),
                            "seeds_better": wide.gt(wide[ref], axis=0).sum()}).round(0)
    return summary.sort_values("mean", ascending=False), wide

## 1. Population settings of v1 (existing knobs only)

In [ ]:
SEEDS = ["2000:2016"]
df_pop = sweep("original-eat-rest-preserved", [
    "base",
    "floor4:population_floor=4",
    "floor6:population_floor=6",
    "decay1800:population_decay=1800",
    "pop9slow:population=9,population_decay=1800,population_floor=4",
], SEEDS, "logs/sweep_population.csv")
summary, wide = table(df_pop); summary

## 2. New rule: only give birth next to food (`eat-rest-fedbirth`)

`off` is the rule disabled and is byte-for-byte the baseline's behaviour, so it doubles as the reference.

In [ ]:
df_fed = sweep("eat-rest-fedbirth", [
    "off:birth_food_distance=0",
    "d100:birth_food_distance=100",
    "d150:birth_food_distance=150",
    "d250:birth_food_distance=250",
    "d150late:birth_food_distance=150,birth_food_start=600",
    "d150pat25:birth_food_distance=150,birth_food_patience=25",
], SEEDS, "logs/sweep_fedbirth.csv")
summary, wide = table(df_fed); summary

## 3. Confirmation on fresh seeds

Put the one or two best variants here (keep the reference first). These seeds were not used to pick them.

In [ ]:
CONFIRM_SEEDS = ["2100:2148"]   # 48 fresh seeds
df_confirm = sweep("eat-rest-fedbirth", [
    "off:birth_food_distance=0",
    "d150:birth_food_distance=150",      # <- replace with the winner(s) from above
], CONFIRM_SEEDS, "logs/sweep_confirm.csv")
summary, wide = table(df_confirm); summary